# Hindi Spelling Classification Pipeline

A comprehensive multi-layer pipeline to classify ~1,75,000 Hindi words as **correct spelling** or **incorrect spelling**.

**Pipeline Steps:**
1. Data Collection & URL Fixing
2. Rebuild Vocabulary from FT Dataset
3. Text Normalization Pipeline
4. High-Frequency Confidence Rule
5. Protected Core Vocabulary
6. Dictionary-Based Validation
7. Orthographic Rule Checking
8. Edit-Distance Typo Detection
9. English-in-Devanagari Handling
10. Conservative Default Policy
11. Final Classification Logic
12. Output Generation
13. Final Count Reporting

In [ ]:
!pip install openpyxl python-Levenshtein pandas requests tqdm -q

In [ ]:
import pandas as pd
import unicodedata
import re
import string
import json
import requests
import Levenshtein
from collections import Counter
from tqdm import tqdm
import warnings
import os
import time
warnings.filterwarnings('ignore')

print("All imports successful!")

## Configuration

Update the file paths below to match your setup.

In [ ]:

FT_DATASET_FILE = "your_ft_dataset.xlsx"       
UNIQUE_WORDS_FILE = "Unique Words Data.xlsx"    
WORD_COLUMN = "word"                            
URL_COLUMN = "transcription_url_gcp"                

FREQUENCY_THRESHOLD = 50
EDIT_DISTANCE_MAX = 2

OLD_URL_BASE = "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/"
NEW_URL_BASE = "https://storage.googleapis.com/upload_goai/"

print("Configuration loaded.")

## Step 1: Data Collection & URL Fixing

Since the original links were broken, we programmatically modify URLs following the pattern:
- **Old:** `https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/{user_id}/{recording_id}_transcription.json`
- **New:** `https://storage.googleapis.com/upload_goai/{user_id}/{recording_id}_transcription.json`

Then download all transcription JSON files and extract the cleaned transcript text.

In [ ]:
def fix_url(url):
    """Fix broken URLs from old format to new format."""
    if not isinstance(url, str):
        return url
    return url.replace(OLD_URL_BASE, NEW_URL_BASE)


def download_transcription(url, retries=3):
    """Download a transcription JSON and extract text."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                
                if isinstance(data, dict):
                    text = data.get("text", data.get("transcript", data.get("transcription", "")))
                elif isinstance(data, list):
                    text = " ".join([item.get("text", "") for item in data if isinstance(item, dict)])
                else:
                    text = str(data)
                return text.strip()
            elif resp.status_code == 404:
                return None
        except Exception as e:
            if attempt == retries - 1:
                return None
            time.sleep(1)
    return None


try:
    ft_df = pd.read_excel(FT_DATASET_FILE, engine="openpyxl")
    print(f"FT Dataset loaded: {len(ft_df)} rows")
    print(f"Columns: {list(ft_df.columns)}")
    
    url_cols = [col for col in ft_df.columns if ft_df[col].dtype == "object"]
    total_fixed = 0
    for col in url_cols:
        mask = ft_df[col].astype(str).str.contains(OLD_URL_BASE, na=False)
        count = mask.sum()
        if count > 0:
            ft_df[col] = ft_df[col].apply(fix_url)
            total_fixed += count
            print(f"  Fixed {count} URLs in column: {col}")
    print(f"Total URLs fixed: {total_fixed}")
    
    if URL_COLUMN in ft_df.columns:
        url_col = URL_COLUMN
    else:
        url_col = None
        for col in ft_df.columns:
            if ft_df[col].astype(str).str.contains("https://", na=False).sum() > 0:
                url_col = col
                break
    
    if url_col:
        print(f"\nURL column identified: {url_col}")
        print(f"Sample fixed URLs:")
        print(ft_df[url_col].dropna().head(3).tolist())
    else:
        print("\nNo URL column found. Skipping download step.")
        
except FileNotFoundError:
    print(f"FT Dataset file not found: {FT_DATASET_FILE}")
    print("Skipping Step 1. Will proceed with Unique Words file only.")
    ft_df = None
    url_col = None

### Download Transcription JSONs

Download all transcription files and extract full cleaned transcript text to reconstruct the corpus.

In [ ]:

all_transcripts = []

if ft_df is not None and url_col is not None:
    urls = ft_df[url_col].dropna().unique()
    print(f"Downloading {len(urls)} transcription files...")

    success = 0
    failed = 0

    for url in tqdm(urls, desc="Downloading"):
        text = download_transcription(str(url))
        if text:
            all_transcripts.append(text)
            success += 1
        else:
            failed += 1

    print(f"\nDownload complete!")
    print(f"  Successful: {success}")
    print(f"  Failed: {failed}")
    print(f"  Total transcript text collected: {sum(len(t) for t in all_transcripts):,} characters")
else:
    print("No FT dataset/URL column available. Skipping download.")
    print("Will use Unique Words file directly.")

## Step 2: Rebuild Vocabulary from FT Dataset

From all downloaded transcriptions:
- Tokenize text into words
- Normalize tokens
- Generate frequency dictionary
- Cross-validate with Unique Words Data

In [ ]:

corpus_word_freq = Counter()

if all_transcripts:
    for text in all_transcripts:
        
        tokens = re.findall(r"[\u0900-\u097F]+", text)
        corpus_word_freq.update(tokens)
    
    print(f"Corpus vocabulary size: {len(corpus_word_freq):,} unique tokens")
    print(f"Total token count: {sum(corpus_word_freq.values()):,}")
    print(f"\nTop 20 most frequent words:")
    for word, freq in corpus_word_freq.most_common(20):
        print(f"  {word:20s} {freq:>8,}")
else:
    print("No transcripts downloaded. Frequency will be computed from unique words file.")

unique_df = pd.read_excel(UNIQUE_WORDS_FILE, engine="openpyxl")
print(f"\nUnique Words file loaded: {len(unique_df)} rows")
print(f"Columns: {list(unique_df.columns)}")

assert WORD_COLUMN in unique_df.columns, f"Column '{WORD_COLUMN}' not found!"
unique_df[WORD_COLUMN] = unique_df[WORD_COLUMN].astype(str)

if corpus_word_freq:
    unique_df["frequency"] = unique_df[WORD_COLUMN].map(
        lambda w: corpus_word_freq.get(w, 0)
    )
    matched = (unique_df["frequency"] > 0).sum()
    print(f"\nCross-validation: {matched}/{len(unique_df)} words found in corpus ({100*matched/len(unique_df):.1f}%)")
else:
    word_counts = Counter(unique_df[WORD_COLUMN])
    unique_df["frequency"] = unique_df[WORD_COLUMN].map(word_counts)
    print("Using word counts from unique words file as frequency.")

df = unique_df.copy()
print(f"\nWorking dataset: {len(df)} words")
print(f"Sample: {df[WORD_COLUMN].head(10).tolist()}")

## Step 3: Text Normalization Pipeline

Before spelling validation:
- Unicode normalize (NFC)
- Strip whitespace
- Remove invisible characters (zero-width spaces, joiners, BOM)
- Normalize nukta variants
- Standardize punctuation removal

This prevents artificial duplicates.

In [ ]:
INVISIBLE_CHARS = re.compile(
    '[\u200b\u200c\u200d\u200e\u200f\u00ad\ufeff\u2060]'
)

NUKTA_MAP = {
    '\u0958': '\u0915\u093C',
    '\u0959': '\u0916\u093C',
    '\u095A': '\u0917\u093C',
    '\u095B': '\u091C\u093C',
    '\u095C': '\u0921\u093C',
    '\u095D': '\u0922\u093C',
    '\u095E': '\u092B\u093C',
    '\u095F': '\u092F\u093C',
}

def normalize_word(word):
    """Full normalization pipeline for a single word."""
    if not isinstance(word, str):
        return str(word)
    word = word.strip()
    word = unicodedata.normalize("NFC", word)
    word = word.strip(string.punctuation + '\u0964\u0965\u0970')
    for nukta_char, replacement in NUKTA_MAP.items():
        word = word.replace(nukta_char, replacement)
    word = INVISIBLE_CHARS.sub("", word)
    return word

df['word_normalized'] = df[WORD_COLUMN].apply(normalize_word)
df = df[df['word_normalized'].str.len() > 0].reset_index(drop=True)

print(f"After normalization: {len(df)} words")
print(f"Sample normalized: {df['word_normalized'].head(10).tolist()}")

## Step 4: High-Frequency Confidence Rule

**Key linguistic assumption:** Genuine spelling mistakes are unlikely to appear frequently.

If `frequency[word] > FREQ_THRESHOLD` -> `correct spelling`

This prevents overcorrection of common words like: है, तो, में, की, नहीं

In [ ]:
if corpus_word_freq:
    df['frequency'] = df['word_normalized'].map(
        lambda w: corpus_word_freq.get(w, 0)
    )

HIGH_CONFIDENCE_WORDS = set(
    df[df['frequency'] > FREQUENCY_THRESHOLD]['word_normalized'].unique()
)

print(f"Frequency threshold: {FREQUENCY_THRESHOLD}")
print(f"High-confidence words (freq > threshold): {len(HIGH_CONFIDENCE_WORDS)}")
print(f"\nSample high-frequency words:")
for w in list(HIGH_CONFIDENCE_WORDS)[:15]:
    print(f"  {w} (freq: {corpus_word_freq.get(w, 'N/A')})")

## Step 5: Protected Core Vocabulary

Curated sets of:
- Hindi stopwords
- Pronouns
- Auxiliary verbs
- Postpositions
- Conversational markers

If word is in this set -> `correct spelling`. Prevents false positives.

In [ ]:
PROTECTED_WORDS = {
    
    "और", "या", "तथा", "एवं", "परन्तु", "लेकिन", "किन्तु", "मगर", "पर",
    "अगर", "यदि", "तो", "फिर", "भी", "ही", "जो", "जब", "तब", "कब",
    "कि", "क्योंकि", "इसलिए", "अतः", "चूँकि", "ताकि", "जैसे", "वैसे",
    "न", "ना", "मत", "नहीं", "नही", "बिना",
    
    "मैं", "मैंने", "मुझे", "मुझको", "मेरा", "मेरी", "मेरे", "हम", "हमने",
    "हमें", "हमारा", "हमारी", "हमारे", "तू", "तूने", "तुझे", "तेरा", "तेरी",
    "तुम", "तुमने", "तुम्हें", "तुम्हारा", "तुम्हारी", "तुम्हारे",
    "आप", "आपने", "आपको", "आपका", "आपकी", "आपके",
    "यह", "वह", "ये", "वे", "इसने", "उसने", "इन्होंने", "उन्होंने",
    "इसे", "उसे", "इसको", "उसको", "इसका", "उसका", "इसकी", "उसकी",
    "कोई", "कुछ", "सब", "सभी", "हर", "प्रत्येक", "कई", "बहुत",
    "खुद", "अपना", "अपनी", "अपने", "स्वयं",
    
    "है", "हैं", "हो", "हूँ", "था", "थी", "थे", "थीं",
    "होगा", "होगी", "होंगे", "होंगी",
    "रहा", "रही", "रहे", "रहें",
    "सकता", "सकती", "सकते", "सकें",
    "चाहिए", "पड़ता", "पड़ती", "पड़ते", "पड़ा", "पड़ी", "पड़े",
    "गया", "गयी", "गई", "गए", "गये",
    "दिया", "दी", "दिए", "दिये",
    "लिया", "ली", "लिए", "लिये",
    "करना", "करता", "करती", "करते", "किया", "की", "किए", "किये",
    "कर", "करो", "करें", "कीजिए",
    "जाना", "जाता", "जाती", "जाते", "जाओ", "जाइए",
    "आना", "आता", "आती", "आते", "आया", "आई", "आए", "आये",
    "देना", "देता", "देती", "देते", "दो", "दीजिए",
    "लेना", "लेता", "लेती", "लेते", "लो", "लीजिए",
    "बोलना", "बोलता", "बोलती", "बोलते", "बोला", "बोली", "बोले",
    "कहना", "कहता", "कहती", "कहते", "कहा", "कही", "कहे",
    "होना", "हुआ", "हुई", "हुए", "हुये",
    "पाना", "चुका", "चुकी", "चुके",

    "में", "से", "को", "का", "के",
    "तक", "द्वारा", "साथ", "बाद", "पहले", "बीच",
    "ऊपर", "नीचे", "अंदर", "बाहर", "पास", "दूर", "सामने", "पीछे",
    "हेतु", "वास्ते", "खातिर",
    "बारे", "विषय", "संबंध", "अनुसार", "मुताबिक",

    "जी", "हाँ", "हां", "अच्छा", "ठीक", "बस",
    "मतलब", "यानी", "अर्थात", "दरअसल", "वास्तव",
    "शायद", "संभवतः", "अवश्य", "ज़रूर", "जरूर",
    "कम", "ज्यादा", "अधिक", "कितना", "कितनी", "कितने",
    "अभी", "कभी", "सदा", "हमेशा", "कल", "आज", "परसों",
    "यहाँ", "वहाँ", "कहाँ", "जहाँ", "यहां", "वहां", "कहां", "जहां",
    "क्या", "कैसे", "कैसा", "कैसी", "क्यों", "कौन", "किसने", "किसको",
    "एक", "दो", "तीन", "चार", "पाँच", "छह", "सात", "आठ", "नौ", "दस",
    "सौ", "हज़ार", "लाख", "करोड़",
    "वाला", "वाली", "वाले",
    "लोग", "लोगों", "देश", "दुनिया", "समय", "बात", "काम", "दिन", "रात",
    "घर", "पानी", "खाना", "पैसा", "जगह", "तरह",
    "सरकार", "भारत", "हिंदी", "हिन्दी",
}

print(f"Protected vocabulary size: {len(PROTECTED_WORDS)} words")

## Step 6: Dictionary-Based Validation

Standard Hindi lexicon corpus and open-source Hindi word lists.
If word in dictionary -> `correct spelling`. If not found -> deeper inspection.

In [ ]:
HINDI_DICTIONARY = {
    
    "व्यक्ति", "इंसान", "मनुष्य", "पुरुष", "महिला", "स्त्री", "बालक", "बालिका",
    "माता", "पिता", "भाई", "बहन", "पत्नी", "पति", "बेटा", "बेटी",
    "दादा", "दादी", "नाना", "नानी", "चाचा", "चाची", "मामा", "मामी",
    "मित्र", "दोस्त", "साथी", "शिक्षक", "विद्यार्थी", "छात्र", "छात्रा",
    "डॉक्टर", "वकील", "इंजीनियर", "किसान", "मज़दूर", "मजदूर", "व्यापारी",
    "नेता", "मंत्री", "राष्ट्रपति", "प्रधानमंत्री", "अधिकारी", "कर्मचारी",
    "पड़ोसी", "अतिथि", "मेहमान", "रिश्तेदार", "परिवार", "समाज",
    
    "शहर", "गाँव", "गांव", "नगर", "राज्य", "प्रदेश", "ज़िला", "जिला",
    "मोहल्ला", "बाज़ार", "बाजार", "दुकान", "स्कूल", "विद्यालय", "विश्वविद्यालय",
    "अस्पताल", "मंदिर", "मस्जिद", "गुरुद्वारा", "चर्च",
    "सड़क", "रास्ता", "मार्ग", "पुल", "नदी", "पहाड़", "समुद्र", "झील",
    "खेत", "बगीचा", "उद्यान", "जंगल", "वन",
    "कार्यालय", "दफ्तर", "संसद", "न्यायालय", "थाना",
    
    "किताब", "पुस्तक", "कागज़", "कागज", "कलम", "पत्र", "अखबार", "समाचार",
    "फ़ोन", "फोन", "गाड़ी", "कुर्सी", "मेज़", "मेज", "बिस्तर",
    "दरवाज़ा", "दरवाजा", "खिड़की", "कपड़ा", "कपड़े",
    "ज्ञान", "शिक्षा", "विज्ञान", "गणित", "इतिहास", "भूगोल", "साहित्य",
    "संस्कृति", "परंपरा", "धर्म", "राजनीति", "अर्थव्यवस्था", "विकास",
    "स्वास्थ्य", "रोग", "बीमारी", "दवाई", "इलाज", "चिकित्सा",
    "प्रेम", "प्यार", "दया", "करुणा", "क्रोध", "गुस्सा", "खुशी", "दुख",
    "शांति", "युद्ध", "संघर्ष", "स्वतंत्रता", "अधिकार", "कर्तव्य", "न्याय",
    "सत्य", "असत्य", "झूठ", "सच",
    
    "पीना", "सोना", "जागना", "उठना", "बैठना", "चलना", "दौड़ना",
    "देखना", "सुनना", "पढ़ना", "लिखना", "गिनना", "सीखना", "सिखाना",
    "समझना", "समझाना", "सोचना", "भूलना",
    "रोना", "हँसना", "हंसना", "मुस्कुराना", "गाना",
    "खेलना", "नाचना", "तैरना", "उड़ना", "कूदना",
    "मारना", "काटना", "तोड़ना", "जोड़ना", "बनाना",
    "खोलना", "धोना", "पकाना", "सुखाना",
    "रखना", "उठाना", "गिराना", "फेंकना", "पकड़ना", "छोड़ना",
    "मिलना", "मनाना", "रुकना", "ठहरना",
    "भेजना", "लाना", "बुलाना", "बताना", "पूछना",
    "जानना", "पहचानना", "चुनना", "बदलना", "हटाना", "लगाना",
    "रोकना", "बचाना", "डरना", "डराना", "माँगना", "मांगना", "कमाना",
    
    "सुंदर", "सुन्दर", "मोटा", "पतला", "लंबा", "लम्बा",
    "ऊँचा", "ऊंचा", "नीचा", "गहरा", "चौड़ा",
    "गर्म", "ठंडा", "ठण्डा", "सूखा", "गीला", "कड़ा", "नरम", "मुलायम",
    "साफ़", "साफ", "गंदा", "गन्दा",
    "तेज़", "तेज", "धीमा",
    "सफ़ेद", "सफेद", "काला", "लाल", "पीला", "हरा", "नीला",
    "मीठा", "खट्टा", "नमकीन", "कड़वा", "तीखा",
    "सही", "ग़लत", "गलत", "उचित", "अनुचित",
    "भारतीय", "विदेशी", "राष्ट्रीय", "सरकारी", "सामाजिक", "आर्थिक",
    
    "पेड़", "पौधा", "फूल", "पत्ता", "बीज", "फल",
    "आसमान", "आकाश", "सूरज", "सूर्य", "चाँद", "चांद", "तारा",
    "बादल", "बारिश", "वर्षा", "बर्फ", "हवा", "धूप",
    "कुत्ता", "बिल्ली", "गाय", "भैंस", "घोड़ा", "शेर", "हाथी",
    
    "रोटी", "चावल", "दाल", "सब्ज़ी", "सब्जी", "आटा", "चीनी", "नमक",
    "दूध", "दही", "घी", "मक्खन", "तेल", "मसाला",
    "आम", "सेब", "केला", "संतरा", "अंगूर", "अनार",
    "आलू", "प्याज़", "प्याज", "टमाटर", "गोभी", "मटर",
    "चाय", "शरबत", "लस्सी",
    
    "सवाल", "जवाब", "समस्या", "समाधान", "उपाय",
    "कारण", "परिणाम", "नतीजा", "प्रभाव", "फायदा", "नुकसान",
    "तरीका", "विधि", "प्रक्रिया", "योजना", "नीति", "कार्यक्रम",
    "जानकारी", "सूचना", "खबर", "रिपोर्ट", "विवरण",
    "मदद", "सहायता", "सहयोग", "सेवा", "सुविधा",
    "कोशिश", "प्रयास", "मेहनत", "परिश्रम",
    "उम्मीद", "आशा", "विश्वास", "भरोसा",
    "अनुभव", "अभ्यास", "परीक्षा",
    "भाषा", "शब्द", "वाक्य", "अर्थ", "परिभाषा",
    "उत्साह", "प्रेरणा", "हिम्मत", "साहस", "धैर्य",
}

print(f"Hindi dictionary size: {len(HINDI_DICTIONARY)} words")

## Step 7: Orthographic Rule Checking

Rule-based validators for:
- **7.1** Invalid matra patterns (vowel before consonant, double matra errors)
- **7.2** Repeated character noise (3+ identical chars, duplicate nasalization)
- **7.3** Invalid consonant clusters

If `violates_hindi_orthography(word)` -> `incorrect spelling`

In [ ]:
CONSONANTS = set(range(0x0915, 0x093A))
VOWELS = set(range(0x0904, 0x0915))
MATRAS = set(range(0x093E, 0x094D))
VIRAMA = 0x094D

def violates_hindi_orthography(word):
    """Check for invalid Devanagari orthographic patterns."""
    if not word:
        return False
    cps = [ord(c) for c in word]
    
    if cps[0] in MATRAS:
        return True
    
    for i in range(len(cps) - 1):
        if cps[i] in MATRAS and cps[i+1] in MATRAS and cps[i] != VIRAMA:
            return True

    for i in range(len(cps) - 1):
        if cps[i] in MATRAS and cps[i] == cps[i+1]:
            return True
    return False


def has_excessive_repetition(word):
    """Detect 3+ consecutive repeated chars or duplicate matras."""
    if not word:
        return False
    count = 1
    for i in range(1, len(word)):
        if word[i] == word[i-1]:
            count += 1
            if count >= 3:
                return True
        else:
            count = 1
    for i in range(len(word) - 1):
        cp1, cp2 = ord(word[i]), ord(word[i+1])
        if cp1 in MATRAS and cp1 == cp2:
            return True
    return False


print("Orthographic rule tests:")
test_words = ["नमस्ते", "भारत", "नहीीी", "अच्छछछ"]
for w in test_words:
    o = violates_hindi_orthography(w)
    r = has_excessive_repetition(w)
    print(f"  {w:15s} -> ortho_violation={o}, repetition={r}")

## Step 8: Edit-Distance Typo Detection

For words not found in dictionary:
- Find nearest valid word
- Compute Levenshtein distance
- If distance <= 2 -> `incorrect spelling` (transcription typo)

Examples: नही->नहीं, मैन->मैं, कम्प्यटर->कंप्यूटर

In [ ]:
def find_nearest_dict_word(word, dictionary, max_distance=2):
    """Find nearest dictionary word within max edit distance."""
    best_match = None
    best_dist = max_distance + 1
    for dict_word in dictionary:
        if abs(len(dict_word) - len(word)) > max_distance:
            continue
        dist = Levenshtein.distance(word, dict_word)
        if dist < best_dist:
            best_dist = dist
            best_match = dict_word
        if dist == 1:
            break
    if best_dist <= max_distance:
        return best_match, best_dist
    return None, None

ALL_VALID_WORDS = HINDI_DICTIONARY | PROTECTED_WORDS | LOANWORD_LIST

print(f"Total valid word pool for edit distance: {len(ALL_VALID_WORDS)} words")

## Step 9: English-in-Devanagari Handling

English spoken words written in Devanagari are considered **correct spelling**.
Built a transliterated English lexicon.

In [ ]:
LOANWORD_LIST = {
    
    "कंप्यूटर", "कम्प्यूटर", "लैपटॉप", "मोबाइल", "फ़ोन", "फोन",
    "स्मार्टफोन", "टैबलेट", "इंटरनेट", "वेबसाइट",
    "सॉफ्टवेयर", "हार्डवेयर", "ऐप", "एप्लिकेशन", "डाटा", "डेटा",
    "सर्वर", "नेटवर्क", "वाईफाई", "ब्लूटूथ",
    "ईमेल", "मैसेज", "चैट", "ऑनलाइन", "ऑफलाइन",
    "गूगल", "फेसबुक", "ट्विटर", "इंस्टाग्राम", "व्हाट्सएप",
    "प्रोग्राम", "कोड", "प्रोग्रामिंग", "डिजिटल", "टेक्नोलॉजी",
    "डाउनलोड", "अपलोड", "अपडेट", "वायरस", "पासवर्ड",
    
    "कॉलेज", "यूनिवर्सिटी", "क्लास", "टीचर", "प्रोफेसर",
    "एग्जाम", "रिजल्ट", "डिग्री", "सर्टिफिकेट",
    "सिलेबस", "प्रोजेक्ट", "असाइनमेंट", "लेक्चर",
    
    "बैंक", "अकाउंट", "लोन", "क्रेडिट", "डेबिट", "चेक",
    "बिजनेस", "ऑफिस", "मैनेजर", "मार्केट", "शॉपिंग",
    "सैलरी", "इनकम", "टैक्स", "बिल", "बजट",
    
    "ट्रेन", "बस", "टैक्सी", "ऑटो", "मेट्रो", "एयरपोर्ट", "स्टेशन",
    "टिकट", "पार्किंग", "पेट्रोल", "डीजल", "इंजन",
    
    "क्रिकेट", "फुटबॉल", "हॉकी", "टेनिस", "बैडमिंटन",
    "मैच", "टीम", "प्लेयर", "कैप्टन", "कोच", "स्टेडियम",
    
    "हॉस्पिटल", "नर्स", "सर्जरी", "ऑपरेशन", "इंजेक्शन", "वैक्सीन",
    "टेस्ट", "स्कैन", "थेरेपी", "कैप्सूल", "एंटीबायोटिक",
    
    "फिल्म", "सिनेमा", "एक्टर", "डायरेक्टर",
    "म्यूजिक", "गिटार", "पियानो", "ड्रम",
    "टीवी", "टेलीविजन", "रेडियो", "चैनल",
    "वीडियो", "यूट्यूब", "नेटफ्लिक्स",
    
    "पार्टी", "इवेंट", "फंक्शन", "सिस्टम", "सर्विस",
    "लिस्ट", "नंबर", "टाइम", "फॉर्म", "कॉपी", "फाइल",
    "ग्रुप", "मेंबर", "लीडर", "इंटरव्यू",
    "प्लान", "टारगेट", "गोल", "स्ट्रेटेजी",
    "रेस्टोरेंट", "होटल", "कॉफी", "बर्गर", "चॉकलेट",
    "बिस्किट", "केक", "आइसक्रीम",
}

print(f"Loanword list size: {len(LOANWORD_LIST)} words")

ALL_VALID_WORDS = HINDI_DICTIONARY | PROTECTED_WORDS | LOANWORD_LIST
print(f"Total valid word pool: {len(ALL_VALID_WORDS)} words")

## Steps 10 & 11: Conservative Default + Final Classification Logic

**Decision priority:**
1. Protected word -> `correct spelling`
2. High frequency -> `correct spelling`
3. Dictionary match -> `correct spelling`
4. Loanword match -> `correct spelling`
5. Orthographic violation -> `incorrect spelling`
6. Excessive repetition -> `incorrect spelling`
7. Edit-distance close to valid word -> `incorrect spelling`
8. **Default (conservative)** -> `correct spelling`

Conservative policy: if no structural anomaly, no violation, not a known typo -> default correct.

In [ ]:
def classify_word(word, word_normalized, frequency):
    """Classify word as correct/incorrect spelling with reason."""
    w = word_normalized
    
    if w in PROTECTED_WORDS:
        return "correct spelling", "protected_word"
    
    if frequency > FREQUENCY_THRESHOLD:
        return "correct spelling", "high_frequency"
    
    if w in HINDI_DICTIONARY:
        return "correct spelling", "dictionary_match"
    
    if w in LOANWORD_LIST:
        return "correct spelling", "loanword_match"
    
    if violates_hindi_orthography(w):
        return "incorrect spelling", "orthographic_violation"
    
    if has_excessive_repetition(w):
        return "incorrect spelling", "excessive_repetition"
    
    nearest, dist = find_nearest_dict_word(w, ALL_VALID_WORDS, max_distance=EDIT_DISTANCE_MAX)
    if nearest is not None and dist <= EDIT_DISTANCE_MAX:
        return "incorrect spelling", f"typo_dist_{dist}_near_{nearest}"
    
    return "correct spelling", "default_conservative"

print("=" * 60)
print("RUNNING CLASSIFICATION PIPELINE")
print("=" * 60)
print("This may take a while for large datasets (edit-distance is O(n*m))\n")

unique_words = df[[WORD_COLUMN, 'word_normalized', 'frequency']].drop_duplicates(subset='word_normalized')
print(f"Unique normalized words to classify: {len(unique_words):,}")

results = []
total = len(unique_words)

for idx, row in tqdm(unique_words.iterrows(), total=total, desc='Classifying'):
    label, reason = classify_word(row[WORD_COLUMN], row['word_normalized'], row['frequency'])
    results.append({
        'word': row[WORD_COLUMN],
        'word_normalized': row['word_normalized'],
        'label': label,
        'reason': reason,
        'frequency': row['frequency']
    })

results_df = pd.DataFrame(results)
print(f"\nClassification complete!")
print(f"\nLabel distribution:")
print(results_df['label'].value_counts())

## Step 12: Output Generation

Generate structured output with `word | label` format.
Export as:
- XLSX (2 sheets: simple + detailed)
- CSV backup

In [ ]:
output_df = results_df[['word', 'label']].copy()

print("Output preview:")
print(output_df.head(20).to_string(index=False))

CSV_OUTPUT = "spelling_classification_output.csv"
output_df.to_csv(CSV_OUTPUT, index=False, encoding='utf-8-sig')
print(f"\nCSV saved: {CSV_OUTPUT}")

XLSX_OUTPUT = "spelling_classification_output.xlsx"
with pd.ExcelWriter(XLSX_OUTPUT, engine='openpyxl') as writer:
    output_df.to_excel(writer, sheet_name='Classification Results', index=False)
    results_df.to_excel(writer, sheet_name='Detailed Results', index=False)

print(f"XLSX saved: {XLSX_OUTPUT}")
print(f"  Sheet 1: Classification Results (word | label)")
print(f"  Sheet 2: Detailed Results (word | normalized | label | reason | frequency)")

## Step 13: Final Count Reporting

Summary statistics of the classification results.

In [ ]:
print("=" * 70)
print("       HINDI SPELLING CLASSIFICATION - FINAL REPORT")
print("=" * 70)

correct_count = (results_df['label'] == 'correct spelling').sum()
incorrect_count = (results_df['label'] == 'incorrect spelling').sum()
total_count = len(results_df)

print(f"\n  Total unique words processed:          {total_count:>10,}")
print(f"  Total unique CORRECT spelled words:    {correct_count:>10,}")
print(f"  Total unique INCORRECT spelled words:  {incorrect_count:>10,}")
print(f"\n  Correct percentage:                    {100*correct_count/total_count:>9.1f}%")
print(f"  Incorrect percentage:                  {100*incorrect_count/total_count:>9.1f}%")

print(f"\n  {\"-\" * 50}")
print(f"  Breakdown by classification reason:")
print(f"  {\"-\" * 50}")

reason_counts = results_df['reason'].apply(
    lambda x: x.split('_near_')[0] if '_near_' in x else x
).value_counts()

for reason, count in reason_counts.items():
    pct = 100 * count / total_count
    print(f"  {reason:40s} {count:>8,}  ({pct:5.1f}%)")

print(f"\n  {\"-\" * 50}")
print(f"  Output files:")
print(f"    - {CSV_OUTPUT}")
print(f"    - {XLSX_OUTPUT}")
print("=" * 70)
print("\n  Pipeline complete!")